# Extra de S5 · MCP en notebooks · 2. Un MCP con el modelo dentro

**Qué vas a construir:** un servidor MCP que lleva **el modelo de retención embebido**. Le pasas un número de cliente y devuelve la probabilidad de que **vuelva a comprar en los próximos 90 días**, con su tramo y una recomendación.

**Qué vas a aprender:**
- publicar un modelo de ML como *herramienta* de un LLM;
- que el MCP **no carga un archivo**, sino `models:/retencion-90d@champion`: lo que mande el alias en MLflow. Así un **rollback** cambia lo que responde, sin reiniciar nada;
- comprobar la **paridad**: lo que dice el MCP debe ser exactamente lo que da el modelo cargado a mano;
- qué hacer con los errores: cliente que no existe, variable que sobra, texto en vez de número.

> Ojo con la lectura: la probabilidad es de **volver a comprar**. Alta = buena noticia. No es un «riesgo de abandono».

In [1]:
import asyncio, json, os, sys, sqlite3
from pathlib import Path

AQUI = Path.cwd()
RAIZ = next(p for p in [AQUI, *AQUI.parents] if (p / '04-Proyecto-Base').is_dir())    # funciona en tu portátil y en un Codespace
PROYECTO = RAIZ / '04-Proyecto-Base'
BD = PROYECTO / 'data' / 'processed' / 'curso.db'
TRABAJO = AQUI / '_trabajo'
TRABAJO.mkdir(exist_ok=True)
print('Repositorio:', RAIZ.name, '| base de datos:', BD.name, 'existe' if BD.is_file() else 'NO EXISTE (mira el notebook 00)')

Repositorio: curso-ml-ia-desarrollo-produccion | base de datos: curso.db existe


## 1. El modelo que manda ahora

Cada entrenamiento registra una versión nueva en MLflow. El alias `champion` es un puntero a «la versión que manda». Si todavía no hay un `champion` (lo normal si aún no hiciste el laboratorio de MLOps), el notebook promueve la última versión para poder trabajar.

In [2]:
sys.path.insert(0, str(PROYECTO))
from ia_mlops import alias_modelo as A, config
uri, mlc = A.conectar()
estado = A.estado(mlc)
CHAMPION_PUESTO_AQUI = 'champion' not in estado['alias']
if CHAMPION_PUESTO_AQUI:
    ultima = max(v['version'] for v in estado['versiones'])
    A.promover(mlc, ultima)
    print(f'No había @champion: lo he puesto en la v{ultima}.\n')
print(A.texto_estado(A.estado(mlc)))

No había @champion: lo he puesto en la v6.

Modelo registrado «retencion-90d»: 6 versiones
  v1   alias: -            valid_auc 0.6982 · test_auc 0.7471  (champion anterior: v6)
  v2   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v3   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v4   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v5   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v6   alias: @champion    valid_auc 0.6982 · test_auc 0.7471
Alias: @champion -> v6


## 2. El servidor

Qué mirar en el código:
- **`modelo_champion()`** pregunta a MLflow a qué versión apunta el alias **en cada llamada** y solo recarga si cambió (la carga es lo caro).
- **`predecir_cliente`** lee las 13 variables del cliente de `fs_produccion` (en solo lectura) y las pasa al modelo. El LLM nunca ve el modelo ni la base de datos.
- **`predecir_variables`** es el «¿y si...?»: un cliente inventado. Valida los nombres y los tipos antes de puntuar.
- **`por_que`** reparte la probabilidad entre las variables. Es exacto porque el modelo es una regresión logística: peso = coeficiente × valor transformado.
- **`tramo_de`** convierte la probabilidad en alta, media o baja con la regla de negocio de `config.py`. **Es una regla de negocio, no sale del modelo.**

In [3]:
%%writefile _trabajo/mcp_modelo.py
"""Servidor MCP «mcp-modelo-retencion»: el modelo de retención embebido; le pasas un cliente y devuelve la predicción.

IDEA
    Un modelo es una función: variables del cliente -> probabilidad. Un MCP la publica como HERRAMIENTA para que un LLM la llame.
    El LLM no ve el modelo ni los datos: pide `predecir_cliente(12369)` y recibe un JSON con la probabilidad, el tramo y la versión.

QUÉ MODELO USA (y por qué eso es MLOps)
    No carga un archivo fijo: carga `models:/retencion-90d@champion`, es decir, «la versión que MANDA ahora» en el registro de MLflow.
    En cada llamada pregunta a qué versión apunta el alias y, si ha cambiado, recarga. Así un rollback (mover `champion` a la versión
    anterior) cambia lo que responde el MCP SIN reiniciarlo y sin tocar el código.

QUÉ PREDICE
    La probabilidad de que el cliente VUELVA A COMPRAR en los próximos 90 días. Alta = buena noticia; no es un riesgo de abandono.

HERRAMIENTAS
    predecir_cliente(cliente_id)     probabilidad con las variables del cliente en fs_produccion (las del último lote)
    predecir_variables(variables)    «¿y si...?»: la probabilidad de un cliente inventado, con sus 13 variables (o las que tengas)
    por_que(cliente_id, n)           las variables que más empujan esa probabilidad (exacto: es una regresión logística)
    modelo_en_uso()                  versión del alias, AUC con que se registró y cuántas versiones hay

ENTORNO: MCP_PROYECTO (carpeta 04-Proyecto-Base), CURSO_DB (curso.db), CURSO_MLFLOW_URI (por defecto models/mlflow.db), MCP_ALIAS (champion).
"""
import math
import os
import sqlite3
import sys
from pathlib import Path

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
sys.path.insert(0, os.environ['MCP_PROYECTO'])                       # para importar ia_mlops.config (variables, nombre del modelo, tramos)

import pandas as pd                                                  # noqa: E402
from mcp.server.fastmcp import FastMCP                               # noqa: E402

from ia_mlops import config                                          # noqa: E402

NOMBRE = 'mcp-modelo-retencion'
ALIAS = os.environ.get('MCP_ALIAS', 'champion')
URI = os.environ.get('CURSO_MLFLOW_URI') or 'sqlite:///' + (config.MODELOS / 'mlflow.db').as_posix()
BD = Path(os.environ.get('CURSO_DB', config.RAIZ / 'data' / 'processed' / 'curso.db'))
FEATURES = list(config.FEATURES)

servidor = FastMCP(NOMBRE, instructions=(
    'Predice si un cliente del caso Retail volverá a comprar en los próximos 90 días. La probabilidad es de VOLVER A COMPRAR: alta = buena noticia, '
    'no es un riesgo de abandono. Responde con lo que devuelvan las herramientas; si devuelven «error», díselo al usuario y no inventes cifras.'))
_cache = {'version': None, 'modelo': None, 'metricas': {}}


def _cliente_mlflow():
    import mlflow
    mlflow.set_tracking_uri(URI)
    return mlflow.MlflowClient()


def modelo_champion():
    """(modelo, versión, métricas) de lo que apunte el alias AHORA. Solo recarga si la versión cambió."""
    c = _cliente_mlflow()
    try:
        mv = c.get_model_version_by_alias(config.MODELO_REGISTRADO, ALIAS)
    except Exception as e:                                           # noqa: BLE001 - MLflow lanza varios tipos
        raise LookupError(f'«{config.MODELO_REGISTRADO}» no tiene el alias @{ALIAS}: asígnalo (python -m ia_mlops.alias_modelo promover --version N)') from e
    version = int(mv.version)
    if _cache['version'] != version:
        import mlflow.sklearn
        _cache.update(version=version, modelo=mlflow.sklearn.load_model(f'models:/{config.MODELO_REGISTRADO}/{version}'),
                      metricas=c.get_run(mv.run_id).data.metrics if mv.run_id else {})
    return _cache['modelo'], version, _cache['metricas']


def tramo_de(p):
    for s in config.SEGMENTOS_RECOMPRA:                              # regla de negocio, no del modelo: se acuerda con quien hace la campaña
        if p >= s['desde']:
            return s['segmento'], s['recomendacion']


def _variables_de(cliente_id):
    """Las 13 variables del cliente en fs_produccion (último corte). Solo lectura."""
    con = sqlite3.connect(BD.resolve().as_uri() + '?mode=ro', uri=True)
    con.row_factory = sqlite3.Row
    try:
        f = con.execute('SELECT * FROM fs_produccion WHERE cliente_id = ? ORDER BY fecha_corte DESC LIMIT 1', (cliente_id,)).fetchone()
    finally:
        con.close()
    return None if f is None else dict(f)


def _puntuar(variables):
    modelo, version, metricas = modelo_champion()
    X = pd.DataFrame([{c: (float('nan') if variables.get(c) is None else float(variables[c])) for c in FEATURES}])
    p = float(modelo.predict_proba(X)[:, 1][0])
    tramo, recomendacion = tramo_de(p)
    return p, tramo, recomendacion, version, modelo


def _cid(valor):
    try:
        return int(valor), None
    except (TypeError, ValueError):
        return None, {'error': f'«cliente_id» debe ser un número entero, no «{valor}».'}


@servidor.tool(name='predecir_cliente', description=(
    'Probabilidad de que UN cliente vuelva a comprar en los próximos 90 días, con sus variables del último lote (fs_produccion), el tramo '
    '(alta/media/baja) y la recomendación de negocio. Úsala cuando pregunten por un cliente concreto. Argumento: cliente_id (número).'))
def predecir_cliente(cliente_id: int) -> dict:
    cid, error = _cid(cliente_id)
    if error:
        return error
    fila = _variables_de(cid)
    if fila is None:
        return {'error': f'El cliente {cid} no está en fs_produccion: no hay variables con las que puntuarlo.'}
    try:
        p, tramo, recomendacion, version, _ = _puntuar(fila)
    except LookupError as e:
        return {'error': str(e)}
    return {'cliente_id': cid, 'probabilidad': round(p, 4), 'tramo': tramo, 'recomendacion': recomendacion, 'version_modelo': version,
            'alias': ALIAS, 'corte_de_las_variables': fila['fecha_corte'], 'variables': {c: fila[c] for c in FEATURES}}


@servidor.tool(name='predecir_variables', description=(
    'Probabilidad de volver a comprar para un cliente HIPOTÉTICO, a partir de sus variables: «¿y si compró 3 veces más?». '
    f'Argumento: variables (objeto con {len(FEATURES)} números: {", ".join(FEATURES)}). Si falta alguna, el modelo usa la mediana de entrenamiento y se avisa.'))
def predecir_variables(variables: dict) -> dict:
    if not isinstance(variables, dict) or not variables:
        return {'error': f'Pasa un objeto con las variables: {", ".join(FEATURES)}.'}
    desconocidas = sorted(set(variables) - set(FEATURES))
    if desconocidas:
        return {'error': f'Variables que el modelo no conoce: {desconocidas}. Las válidas: {FEATURES}.'}
    for k, v in variables.items():
        if v is not None and (isinstance(v, bool) or not isinstance(v, (int, float))):
            return {'error': f'«{k}» debe ser un número, no «{v}».'}
        if isinstance(v, float) and not math.isfinite(v):
            return {'error': f'«{k}» no es un número finito.'}
    faltan = [c for c in FEATURES if variables.get(c) is None]
    try:
        p, tramo, recomendacion, version, _ = _puntuar(variables)
    except LookupError as e:
        return {'error': str(e)}
    r = {'probabilidad': round(p, 4), 'tramo': tramo, 'recomendacion': recomendacion, 'version_modelo': version, 'alias': ALIAS}
    if faltan:
        r['aviso'] = f'Faltaban {faltan}: el modelo las imputó con la mediana del entrenamiento.'
    return r


def _pesos(modelo, fila):
    """Regresión logística dentro de un Pipeline: logit = intercepto + Σ coeficiente × valor transformado. Exacto. None si no es de ese tipo."""
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    if not isinstance(modelo, Pipeline) or not isinstance(modelo.steps[-1][1], LogisticRegression):
        return None
    logistica = modelo.steps[-1][1]
    X = pd.DataFrame([{c: (float('nan') if fila.get(c) is None else float(fila[c])) for c in FEATURES}])
    Z = np.asarray(modelo[:-1].transform(X), dtype='float64')[0]
    pesos = logistica.coef_[0] * Z
    logit = float(logistica.intercept_[0]) + float(pesos.sum())
    return {'pesos': pesos, 'intercepto': float(logistica.intercept_[0]), 'logit': logit, 'probabilidad': 1.0 / (1.0 + math.exp(-logit))}


@servidor.tool(name='por_que', description=(
    'Las variables que más empujan la probabilidad de UN cliente, hacia arriba o hacia abajo. Úsala cuando pregunten por qué sale alto o bajo. '
    'Argumentos: cliente_id (número) y n (cuántas variables, por defecto 5).'))
def por_que(cliente_id: int, n: int = 5) -> dict:
    cid, error = _cid(cliente_id)
    if error:
        return error
    fila = _variables_de(cid)
    if fila is None:
        return {'error': f'El cliente {cid} no está en fs_produccion.'}
    try:
        p, tramo, _, version, modelo = _puntuar(fila)
    except LookupError as e:
        return {'error': str(e)}
    d = _pesos(modelo, fila)
    if d is None:
        return {'cliente_id': cid, 'probabilidad': round(p, 4), 'version_modelo': version,
                'nota': 'El modelo no es una regresión logística: no se puede repartir la probabilidad por variable de forma exacta.'}
    if abs(d['probabilidad'] - p) > 1e-6:
        return {'cliente_id': cid, 'probabilidad': round(p, 4), 'version_modelo': version, 'nota': 'La descomposición no reproduce la probabilidad: no se devuelven pesos.'}
    filas = sorted(({'variable': c, 'valor': fila[c], 'peso_logit': round(float(w), 4), 'efecto': 'sube' if w > 0 else 'baja' if w < 0 else 'neutro'}
                    for c, w in zip(FEATURES, d['pesos'])), key=lambda f: (-abs(f['peso_logit']), f['variable']))
    return {'cliente_id': cid, 'probabilidad': round(p, 4), 'tramo': tramo, 'version_modelo': version, 'intercepto_logit': round(d['intercepto'], 4),
            'logit_total': round(d['logit'], 4), 'variables_que_mas_pesan': filas[:max(1, min(int(n), len(FEATURES)))],
            'nota': 'Intercepto + suma de los pesos = logit; probabilidad = 1/(1+e^-logit). Dice qué empuja la cifra, no por qué compra el cliente.'}


@servidor.tool(name='modelo_en_uso', description='Qué versión del modelo responde ahora (a la que apunta el alias), con su AUC de validación y test y cuántas versiones hay. No necesita argumentos.')
def modelo_en_uso() -> dict:
    try:
        _, version, metricas = modelo_champion()
    except LookupError as e:
        return {'error': str(e)}
    versiones = len(_cliente_mlflow().search_model_versions(f"name='{config.MODELO_REGISTRADO}'"))
    return {'modelo': config.MODELO_REGISTRADO, 'alias': ALIAS, 'version': version, 'versiones_registradas': versiones,
            'valid_auc': metricas.get('valid_auc'), 'test_auc': metricas.get('test_auc')}


def main():
    print(f'{NOMBRE}: alias @{ALIAS} en {URI}', file=sys.stderr)      # por stderr: stdout es el canal del protocolo
    servidor.run(transport='stdio')


if __name__ == '__main__':
    main()

Overwriting _trabajo/mcp_modelo.py


## 3. El cliente

La misma clase que en el notebook 01.

In [4]:
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class ClienteMCP:
    """Lanza un servidor MCP como proceso hijo y habla con él por su entrada y salida estándar (transporte «stdio»).
    Es lo mismo que hacen VS Code, Claude Desktop o un agente: arrancan el servidor, le piden las herramientas y las llaman por nombre."""

    def __init__(self, script, **entorno):
        self.params = StdioServerParameters(command=sys.executable, args=[str(script)], env={**os.environ, **{k: str(v) for k, v in entorno.items()}})

    async def __aenter__(self):
        self._pila = AsyncExitStack()
        self._log = open(TRABAJO / 'servidor.err', 'w', encoding='utf-8')          # lo que el servidor escribe por stderr
        leer, escribir = await self._pila.enter_async_context(stdio_client(self.params, errlog=self._log))
        self.sesion = await self._pila.enter_async_context(ClientSession(leer, escribir))
        await self.sesion.initialize()
        return self

    async def __aexit__(self, *a):
        await self._pila.aclose()
        self._log.close()

    async def herramientas(self):
        return (await self.sesion.list_tools()).tools

    async def llamar(self, nombre, **argumentos):
        r = await self.sesion.call_tool(nombre, argumentos)
        if r.isError:                                                                # el propio protocolo rechazó los argumentos (tipos)
            return {'error_del_protocolo': r.content[0].text}
        return r.structuredContent or json.loads(r.content[0].text)

## 4. Qué ofrece

Las descripciones dicen cuándo usar cada herramienta y cuándo no.

In [5]:
ENTORNO = dict(MCP_PROYECTO=PROYECTO, CURSO_DB=BD, CURSO_MLFLOW_URI=uri)
async with ClienteMCP(TRABAJO / 'mcp_modelo.py', **ENTORNO) as c:
    for h in await c.herramientas():
        print(f'• {h.name}\n    {h.description}\n')

• predecir_cliente
    Probabilidad de que UN cliente vuelva a comprar en los próximos 90 días, con sus variables del último lote (fs_produccion), el tramo (alta/media/baja) y la recomendación de negocio. Úsala cuando pregunten por un cliente concreto. Argumento: cliente_id (número).

• predecir_variables
    Probabilidad de volver a comprar para un cliente HIPOTÉTICO, a partir de sus variables: «¿y si compró 3 veces más?». Argumento: variables (objeto con 13 números: recencia_dias, frecuencia, monto_total, ticket_medio, unidades_total, productos_distintos, antiguedad_dias, frecuencia_90d, monto_90d, frecuencia_180d, dias_medios_entre_compras, devoluciones, es_reino_unido). Si falta alguna, el modelo usa la mediana de entrenamiento y se avisa.

• por_que
    Las variables que más empujan la probabilidad de UN cliente, hacia arriba o hacia abajo. Úsala cuando pregunten por qué sale alto o bajo. Argumentos: cliente_id (número) y n (cuántas variables, por defecto 5).

• modelo_en_uso
    

## 5. Predecir un cliente

In [6]:
async with ClienteMCP(TRABAJO / 'mcp_modelo.py', **ENTORNO) as c:
    r = await c.llamar('predecir_cliente', cliente_id=12369)
    print(json.dumps({k: v for k, v in r.items() if k != 'variables'}, ensure_ascii=False, indent=2))
    print('\nvariables que usó el modelo:', r['variables'])
    print('\n--- ¿por qué?')
    p = await c.llamar('por_que', cliente_id=12369, n=4)
    for v in p['variables_que_mas_pesan']:
        print(f'  {v["variable"]:<18} valor {v["valor"]:<8} peso {v["peso_logit"]:+.3f}  → {v["efecto"]} la probabilidad')
    print(f'  intercepto {p["intercepto_logit"]:+.3f}  logit total {p["logit_total"]:+.3f}  →  probabilidad {p["probabilidad"]}')
    print('\n--- el modelo en uso')
    print(await c.llamar('modelo_en_uso'))

{
  "cliente_id": 12369,
  "probabilidad": 0.3035,
  "tramo": "baja",
  "recomendacion": "Poco probable que compre en 90 días: contacto de recuperación de bajo coste o no invertir.",
  "version_modelo": 6,
  "alias": "champion",
  "corte_de_las_variables": "2011-12-10"
}

variables que usó el modelo: {'recencia_dias': 413, 'frecuencia': 3, 'monto_total': 1511.15, 'ticket_medio': 503.72, 'unidades_total': 1234, 'productos_distintos': 89, 'antiguedad_dias': 641, 'frecuencia_90d': 0, 'monto_90d': 0.0, 'frecuencia_180d': 0, 'dias_medios_entre_compras': 114.1, 'devoluciones': 0, 'es_reino_unido': 0}

--- ¿por qué?
  frecuencia_180d    valor 0        peso -1.339  → baja la probabilidad
  recencia_dias      valor 413      peso +0.583  → sube la probabilidad
  devoluciones       valor 0        peso -0.516  → baja la probabilidad
  es_reino_unido     valor 0        peso +0.390  → sube la probabilidad
  intercepto +0.225  logit total -0.831  →  probabilidad 0.3035

--- el modelo en uso
{'modelo'

## 6. ¿Y si...? Un cliente inventado

Partimos del cliente de ejemplo del proyecto y movemos **una sola variable**: la recencia (días desde su última compra). La intuición dice que cuanto más tiempo sin comprar, menos probable que vuelva. **¿Qué dice el modelo?**

In [7]:
import pandas as pd
ejemplo = json.loads((PROYECTO / 'tests' / 'datos' / 'cliente_ejemplo.json').read_text(encoding='utf-8'))
filas = []
async with ClienteMCP(TRABAJO / 'mcp_modelo.py', **ENTORNO) as c:
    for dias in (5, 35, 90, 180, 365):
        r = await c.llamar('predecir_variables', variables={**ejemplo, 'recencia_dias': dias})
        filas.append({'recencia_dias': dias, 'probabilidad': r['probabilidad'], 'tramo': r['tramo']})
    parcial = await c.llamar('predecir_variables', variables={'recencia_dias': 10})
pd.DataFrame(filas)

,recencia_dias,probabilidad,tramo
0,5,0.7533,alta
1,35,0.8279,alta
2,90,0.8589,alta
3,180,0.8787,alta
4,365,0.8965,alta


In [8]:
print('Con solo una variable:', parcial['probabilidad'], '-', parcial['aviso'][:90] + '…')

Con solo una variable: 0.2395 - Faltaban ['frecuencia', 'monto_total', 'ticket_medio', 'unidades_total', 'productos_distin…


**Sorpresa: la probabilidad SUBE con la recencia**, al revés que la intuición. No es un fallo del MCP (lo comprobamos en el apartado 7: da exactamente lo que da el modelo).
Es lo que el modelo aprendió con estos datos; **no es una relación causal**, y con una muestra de 150 clientes y variables muy relacionadas entre sí (recencia, antigüedad y frecuencia) un coeficiente suelto puede engañar.
Tampoco hemos comprobado aquí por qué ocurre. Es una buena pregunta para el análisis de tu tarea. Además, mover **una** variable cada vez no equivale a un cliente real, porque en un cliente real las variables se mueven juntas.

## 7. Paridad: el MCP no inventa nada

Cargamos el modelo **a mano**, sin MCP, y puntuamos a los mismos clientes. Las probabilidades tienen que coincidir (el MCP redondea a 4 decimales).

In [9]:
import mlflow.sklearn
modelo = mlflow.sklearn.load_model(f'models:/{config.MODELO_REGISTRADO}@champion')
con = sqlite3.connect(BD)
ids = [r[0] for r in con.execute('SELECT cliente_id FROM fs_produccion ORDER BY cliente_id LIMIT 12')]
X = pd.read_sql('SELECT * FROM fs_produccion WHERE cliente_id IN (%s)' % ','.join(map(str, ids)), con).set_index('cliente_id')
con.close()
directo = modelo.predict_proba(X.loc[ids, config.FEATURES].astype(float))[:, 1]

async with ClienteMCP(TRABAJO / 'mcp_modelo.py', **ENTORNO) as c:
    por_mcp = [(await c.llamar('predecir_cliente', cliente_id=i))['probabilidad'] for i in ids]

tabla = pd.DataFrame({'cliente_id': ids, 'a mano': directo.round(4), 'por MCP': por_mcp})
tabla['diferencia'] = (tabla['a mano'] - tabla['por MCP']).abs()
print(tabla.to_string(index=False))
assert tabla['diferencia'].max() < 1e-4
print('\nParidad exacta: la diferencia máxima es', tabla['diferencia'].max())

 cliente_id  a mano  por MCP  diferencia
      12369  0.3035   0.3035         0.0
      12425  0.6698   0.6698         0.0
      12430  0.3570   0.3570         0.0
      12503  0.3104   0.3104         0.0
      12513  0.5959   0.5959         0.0
      12523  0.9917   0.9917         0.0
      12567  0.9351   0.9351         0.0
      12642  0.6293   0.6293         0.0
      12654  0.5151   0.5151         0.0
      12665  0.6014   0.6014         0.0
      12670  0.7988   0.7988         0.0
      12692  0.2659   0.2659         0.0

Paridad exacta: la diferencia máxima es 0.0


## 8. Rollback sin reiniciar

Esta es la parte MLOps. Con el servidor **arrancado**, movemos el alias `champion` a otra versión (como harías en un rollback) y volvemos a preguntar. El MCP responde con la versión nueva **sin que lo reinicies ni toques su código**.

> Honestidad: todas las versiones de este curso se entrenaron con los mismos datos, así que **dan la misma probabilidad**. Lo que cambia es la versión que responde; con modelos de verdad distintos cambiaría también la cifra.

In [10]:
estado = A.estado(mlc)
actual = estado['alias']['champion']
otra = next(v['version'] for v in estado['versiones'] if v['version'] != actual)
async with ClienteMCP(TRABAJO / 'mcp_modelo.py', **ENTORNO) as c:
    antes = await c.llamar('predecir_cliente', cliente_id=12369)
    print(f'1. el MCP responde con la v{antes["version_modelo"]}: probabilidad {antes["probabilidad"]}')
    A.promover(mlc, otra)
    despues = await c.llamar('predecir_cliente', cliente_id=12369)
    print(f'2. promovemos @champion a la v{otra}, SIN reiniciar el MCP → responde la v{despues["version_modelo"]}: probabilidad {despues["probabilidad"]}')
    A.volver(mlc)
    vuelta = await c.llamar('predecir_cliente', cliente_id=12369)
    print(f'3. rollback (volver) → responde la v{vuelta["version_modelo"]}: probabilidad {vuelta["probabilidad"]}')
assert (antes['version_modelo'], despues['version_modelo'], vuelta['version_modelo']) == (actual, otra, actual)

1. el MCP responde con la v6: probabilidad 0.3035
2. promovemos @champion a la v1, SIN reiniciar el MCP → responde la v1: probabilidad 0.3035
3. rollback (volver) → responde la v6: probabilidad 0.3035


## 9. Cuando algo sale mal

Un MCP bueno **no se rompe**: devuelve un error claro que el LLM puede contar al usuario. Prueba estas entradas.

In [11]:
MALAS = [('cliente que no existe', 'predecir_cliente', {'cliente_id': 1}),
         ('texto en vez de número', 'predecir_cliente', {'cliente_id': 'abc'}),
         ('variable que el modelo no conoce', 'predecir_variables', {'variables': {'edad': 40}}),
         ('valor que no es número', 'predecir_variables', {'variables': {'frecuencia': 'muchas'}}),
         ('sin variables', 'predecir_variables', {'variables': {}})]
async with ClienteMCP(TRABAJO / 'mcp_modelo.py', **ENTORNO) as c:
    for nombre, herramienta, args in MALAS:
        r = await c.llamar(herramienta, **args)
        print(f'{nombre:<34} → {str(r.get("error") or r.get("error_del_protocolo"))[:110]}')

cliente que no existe              → El cliente 1 no está en fs_produccion: no hay variables con las que puntuarlo.
texto en vez de número             → Error executing tool predecir_cliente: 1 validation error for predecir_clienteArguments
cliente_id
  Input sho
variable que el modelo no conoce   → Variables que el modelo no conoce: ['edad']. Las válidas: ['recencia_dias', 'frecuencia', 'monto_total', 'tick
valor que no es número             → «frecuencia» debe ser un número, no «muchas».
sin variables                      → Pasa un objeto con las variables: recencia_dias, frecuencia, monto_total, ticket_medio, unidades_total, produc


## 10. Conectarlo a un cliente de verdad

```json
{
  "servers": {
    "modelo-retencion": {
      "type": "stdio",
      "command": "python",
      "args": ["01-Sesiones/S5-MLOps-y-Operacion/extra-mcp-notebooks/_trabajo/mcp_modelo.py"],
      "env": { "MCP_PROYECTO": "04-Proyecto-Base", "CURSO_DB": "04-Proyecto-Base/data/processed/curso.db" }
    }
  }
}
```

Con los **dos** servidores a la vez, un agente puede contestar «¿qué cliente del Reino Unido tiene más probabilidad de volver y cuánto ha comprado?»: una pregunta a `mcp-datos-master` y otra a `mcp-modelo-retencion`.

## Ejercicios

1. **Una herramienta nueva** `comparar_clientes(a, b)` que devuelva las dos probabilidades y cuál es mayor.
2. **Un límite:** haz que `predecir_variables` rechace una `recencia_dias` negativa. ¿Dónde pondrías esa regla: en el MCP o en el modelo?
3. **Pensar:** `predecir_cliente` solo ve el último lote (`fs_produccion`). ¿Qué cambiaría si calculase las variables al vuelo, como `puntuar_online` en S4?

In [12]:
if CHAMPION_PUESTO_AQUI:
    print('Este notebook puso @champion al empezar. Si quieres dejar el registro como estaba:')
    print("  mlc.delete_registered_model_alias(config.MODELO_REGISTRADO, 'champion')")

Este notebook puso @champion al empezar. Si quieres dejar el registro como estaba:
  mlc.delete_registered_model_alias(config.MODELO_REGISTRADO, 'champion')
